# 03_train_models

**Owner:** Models and evaluation. **Blueprint step:** 3. **Status:** Runs locally. Not yet run in Fabric.

Trains one model release for the fictional lender HomeBridge. Synthetic data only.

- Rules baseline: product-transfer request friction.
- Logistic regression and a shallow decision tree (depth 3 or 4).
- Two to five behavioural clusters, fitted once per release.

Rules:

- INV-2: snapshots must hold only data known at their cutoff. The run stops otherwise.
- INV-4: only labelled outcomes train the model. Unknown, pending and excluded outcomes never count as retained.
- Training and validation labels must be observed before the simulated release date. Test cohorts end later. Their borrowers stay out of training.
- Hyperparameters and alert thresholds are frozen on validation data.

Inputs: `lakehouse/features/feature_snapshot` and `observed_outcome` from Data and replay. Output: an MLflow run, registered models `hb-refi-clf` and `hb-segments`, and a release folder.

In [ ]:
OUTPUT_ROOT = None  # default: data/output
USE_MLFLOW = True
N_BOOT = 20  # bootstrap fits per k for cluster stability

In [ ]:
import sys
from pathlib import Path

# Local run: this notebook sits in notebooks/ and imports notebooks/models_eval.
# Fabric run (not yet done): upload models_eval as a workspace resource or environment library,
# set HB_OUTPUT_ROOT to the Lakehouse Files path and HB_MLFLOW_URI = 'fabric'.
NOTEBOOKS_DIR = next(p for p in [Path.cwd(), Path.cwd() / 'notebooks'] if (p / 'models_eval').exists())
sys.path.insert(0, str(NOTEBOOKS_DIR))
import json
import pandas as pd
from models_eval import config
PATHS = config.paths(OUTPUT_ROOT)
print('Output root:', PATHS.root)

In [ ]:
from models_eval import train
release = train.train_release(PATHS, use_mlflow=USE_MLFLOW, n_boot=N_BOOT)
{k: release[k] for k in ('release_id', 'champion', 'model_version', 'cluster_model_version', 'alert_threshold', 'registry')}

## Split

Rows per split after the release-date rule and the shared-borrower rule.

In [ ]:
pd.DataFrame({k: release['split'][k] for k in ('rows', 'positive_rate', 'dropped_after_release', 'dropped_shared_borrower')})

## Validation results

The champion is the learned model with the higher validation average precision. Logistic regression wins a near tie. Each threshold maximises F1 on validation data. The team confirms the routing threshold.

In [ ]:
pd.DataFrame(release['validation']).T

In [ ]:
pd.DataFrame(release['params']).T

## Segments

k is the best silhouette among stable solutions (mean bootstrap adjusted Rand index of at least 0.8, worst of at least 0.5). Segment names follow a human inspection of these profiles.

In [ ]:
pd.DataFrame(release['segments']['selection'])

In [ ]:
pd.DataFrame(release['segments']['profiles'])